In [34]:
import pandas as pd
from functools import reduce

In [35]:
# DEfinimos una funciion de estandarizacion que crea la
# fecha en la que inicia cada trimestre para asegurarnos de que este en cada df
# y que los datods con los que haremos el merge sean del mismo tipo en todos los datasets

def preparar_para_merge_blindado(ruta_archivo):
    df = pd.read_csv(ruta_archivo)
    
    #  Limpieza de columnas Unnamed y espacios en nombres
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.columns = df.columns.str.strip() 

    # Limpieza para la columna de trimestres
    df['trimestre'] = df['trimestre'].astype(str).str.replace('*', '', regex=False).str.strip().str.upper()
    df['trimestre'] = df['trimestre'].replace('LL', 'II')
    
    # Forzamos las llaves
    df['anio'] = pd.to_numeric(df['anio'], errors='coerce').ffill().astype(int)
    
    # Crear/Estandarizar Fecha
    mapping_trim = {'I': '01-01', 'II': '04-01', 'III': '07-01', 'IV': '10-01'}
    df['fecha'] = pd.to_datetime(df['anio'].astype(str) + '-' + df['trimestre'].map(mapping_trim))
    
    # Limpiar valores numéricos (por si hay "ND" o "$")
    # Solo aplicamos a columnas que no sean las llaves
    cols_data = [c for c in df.columns if c not in ['fecha', 'anio', 'trimestre']]
    for col in cols_data:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
    return df

In [36]:
# Lista de las rutas de los archivos a mergear
archivos = [
    '../data/clean_data/pobreza_laboral_jefatura_sexo_clean.csv',
    '../data/clean_data/ingreso_promedio_poblacion_ocupada_sexo_clean.csv',
    '../data/clean_data/pobreza_poblacion_ocupada_formalidad_clean.csv',
    '../data/clean_data/pobreza_poblacion_ocupada_sexo_clean.csv'

]

In [37]:
def merge_datasets(left, right):
    llaves = ['fecha', 'anio', 'trimestre']
    # Buscamos qué columnas ya tenemos para no repetirlas
    repetidas = [c for c in right.columns if c in left.columns and c not in llaves]
    
    # Tiramos las repetidas del dataset que entra (right)
    right_clean = right.drop(columns=repetidas)
    
    return pd.merge(left, right_clean, on=llaves, how='outer')

In [38]:
try:
    # Preparamos todos con el mismo formato de datos
    dfs_listos = [preparar_para_merge_blindado(a) for a in archivos]
    
    # Unimos con la lógica de no duplicar columnas
    df_maestro = reduce(merge_datasets, dfs_listos)
    
    # Orden final
    df_maestro = df_maestro.sort_values('fecha').reset_index(drop=True)
    
    # Guardar
    df_maestro.to_csv('../data/dataset_final_tablero.csv', index=False)
    print("Todo unido sin errores.")
    
except Exception as e:
    print(f"The code broke cuz: {e}")

Todo unido sin errores.


In [39]:
df_maestro

,fecha,anio,trimestre,pob_jefatura_nal,pob_jefatura_hombres,pob_jefatura_mujeres,ing_jefatura_nal,ing_jefatura_hombres,ing_jefatura_mujeres,ingreso_hombres,ingreso_mujeres,brecha_salarial,pobreza_total,pobreza_formal,pobreza_informal,ingreso_total,ingreso_formal,ingreso_informal,pobreza_hombres,pobreza_mujeres
0,2005-01-01,2005,I,38.1,36.2,45.2,2605.92,2700.40,2268.47,7421.18,5353.32,0.278643,16.5,0.8,26.8,6677.89,10360.99,4246.08,14.0,20.9
1,2005-04-01,2005,II,38.8,36.4,46.7,2608.59,2710.83,2285.59,7410.56,5356.61,0.277165,17.1,0.9,27.7,6651.84,10347.07,4234.24,14.5,21.6
2,2005-07-01,2005,III,38.1,35.5,46.8,2641.13,2747.83,2271.76,7318.11,5423.90,0.258839,17.6,0.9,28.3,6618.29,10479.46,4151.38,15.0,22.1
3,2005-10-01,2005,IV,36.6,34.4,43.8,2729.02,2838.33,2357.85,7540.13,5587.89,0.258913,16.1,0.7,26.1,6813.61,10674.35,4319.45,13.5,20.6
4,2006-01-01,2006,I,36.2,34.0,43.7,2722.13,2821.10,2386.72,7572.22,5631.87,0.256246,15.6,0.7,25.6,6851.87,10610.62,4331.74,13.1,19.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2023-10-01,2023,IV,37.0,35.3,41.1,3139.61,3251.70,2880.86,7598.45,6013.84,0.208544,12.5,1.0,21.1,6945.33,9827.52,4799.28,10.0,16.2
76,2024-01-01,2024,I,35.8,33.9,40.1,3277.58,3411.01,2966.75,8029.47,6296.22,0.215861,12.0,1.1,20.3,7318.09,10280.68,5051.33,9.2,15.9
77,2024-04-01,2024,II,35.0,33.3,38.9,3350.84,3454.15,3113.58,8137.81,6450.51,0.207341,12.0,0.8,20.4,7441.06,10513.82,5119.51,9.4,15.7
78,2024-07-01,2024,III,35.1,33.2,39.4,3346.45,3470.34,3062.03,8067.53,6433.15,0.202587,12.5,0.7,21.2,7397.27,10583.78,5018.76,9.9,16.1


In [40]:
df_maestro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   fecha                 80 non-null     datetime64[ns]
 1   anio                  80 non-null     int32         
 2   trimestre             80 non-null     object        
 3   pob_jefatura_nal      79 non-null     float64       
 4   pob_jefatura_hombres  79 non-null     float64       
 5   pob_jefatura_mujeres  79 non-null     float64       
 6   ing_jefatura_nal      79 non-null     float64       
 7   ing_jefatura_hombres  79 non-null     float64       
 8   ing_jefatura_mujeres  79 non-null     float64       
 9   ingreso_hombres       79 non-null     float64       
 10  ingreso_mujeres       79 non-null     float64       
 11  brecha_salarial       79 non-null     float64       
 12  pobreza_total         79 non-null     float64       
 13  pobreza_formal        